# Multimodal Emotion Recognition (Colab T4-ready)
This notebook is configured for **Google Colab T4** (set Runtime → Change runtime type → GPU: **T4**).  
It trains a multimodal model (text + image) to predict emotion labels from your merged dataset.

**What you'll find**
- Environment & requirements
- Data validation & loader (handles missing images)
- DeBERTa-v3-base text encoder (from `transformers`)
- EfficientNet-b0 image encoder (from `timm`)
- Fusion classifier (PyTorch)
- Train / validation loops with live plots (loss, accuracy, macro F1)
- Save / load model + simple inference example

**Before running**
1. Upload your `merged_dataset.csv` and `img/` folder to the Colab session (or mount Google Drive and edit paths).
2. Ensure the CSV columns are: `text`, `img` (relative path like `img/23056.png`), `emotion`.


In [1]:
# Install required packages (run once)
# On Colab this will install timm and transformers if not present.
!pip install -q timm transformers torchmetrics --upgrade


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\SRUSANTH PATNAIK\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\torch\\include\\ATen\\native\\transformers\\cuda\\mem_eff_attention\\iterators\\predicated_tile_access_iterator_residual_last.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths



In [2]:
# Imports & configuration
import os
import random
import pandas as pd
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T

from transformers import AutoTokenizer, AutoModel
import timm
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Config (tweak as needed)
DATA_CSV = "merged_dataset.csv"   # path to your merged CSV
IMAGE_ROOT = "img"                # folder prefix in the 'img' column
PRETRAINED_TEXT = "microsoft/deberta-v3-base"
IMAGE_BACKBONE = "tf_efficientnet_b0"
IMG_SIZE = 224
MAX_TEXT_LEN = 128
BATCH_SIZE = 16           # T4 should handle 12-24 depending on model & mixed precision
NUM_EPOCHS = 6
LR = 2e-5
SEED = 42
NUM_WORKERS = 2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


ModuleNotFoundError: No module named 'tqdm'

In [ ]:
# Load CSV and validate image paths
df = pd.read_csv(DATA_CSV)
print('Initial rows:', len(df))
df.head()

# Normalize image paths
def img_path_from_row(x):
    p = str(x)
    if os.path.isabs(p):
        return p
    if p.startswith(IMAGE_ROOT + os.sep) or p.startswith(IMAGE_ROOT + "/"):
        return p
    return os.path.join(IMAGE_ROOT, p.replace("img/", "").lstrip("/"))

df['img_path'] = df['img'].apply(img_path_from_row)

# Count missing images
missing_mask = ~df['img_path'].apply(os.path.exists)
print('Missing images:', missing_mask.sum())
if missing_mask.sum() > 0:
    # drop missing rows
    df = df[~missing_mask].reset_index(drop=True)
    print('After dropping missing:', len(df))

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['emotion'].astype(str))
num_classes = len(le.classes_)
print('Classes:', le.classes_, '->', num_classes)

df.sample(5)


In [ ]:
# Tokenizer & image transforms
tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_TEXT)

image_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print('Tokenizer and image transform ready.')

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform, max_text_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.img_t = image_transform
        self.max_text_len = max_text_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text'])
        enc = self.tokenizer(text,
                             max_length=self.max_text_len,
                             padding='max_length',
                             truncation=True,
                             return_tensors='pt')
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        img_path = row['img_path']
        try:
            img = Image.open(img_path).convert('RGB')
            img = self.img_t(img)
        except Exception as e:
            img = torch.zeros(3, IMG_SIZE, IMG_SIZE)
        label = torch.tensor(int(row['label']), dtype=torch.long)
        return enc, img, label

# Create dataset and splits
dataset = MultimodalDataset(df, tokenizer, image_transform, max_text_len=MAX_TEXT_LEN)
n = len(dataset)
val_ratio = 0.1
n_val = int(n * val_ratio)
n_train = n - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(SEED))
print('Train:', len(train_ds), 'Val:', len(val_ds))

def collate_fn(batch):
    keys = batch[0][0].keys()
    batch_enc = {k: torch.stack([item[0][k] for item in batch]) for k in keys}
    imgs = torch.stack([item[1] for item in batch])
    labels = torch.stack([item[2] for item in batch])
    return batch_enc, imgs, labels

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS)



In [ ]:
# Load text and image backbones
text_model = AutoModel.from_pretrained(PRETRAINED_TEXT)
text_hidden_size = text_model.config.hidden_size
print('Text hidden size:', text_hidden_size)

image_model = timm.create_model(IMAGE_BACKBONE, pretrained=True, num_classes=0, global_pool='avg')
# Try to infer image feature dim
try:
    img_feat_dim = image_model.num_features
except:
    try:
        img_feat_dim = image_model.feature_info[-1]['num_chs']
    except:
        img_feat_dim = 1280
print('Image feat dim:', img_feat_dim)

# Multimodal classifier
class MultimodalClassifier(nn.Module):
    def __init__(self, text_model, image_model, text_dim, img_dim, hidden_dim=512, num_classes=2, dropout=0.3):
        super().__init__()
        self.text_model = text_model
        self.image_model = image_model
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        self.image_proj = nn.Linear(img_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim*2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, enc, img):
        text_out = self.text_model(**enc, return_dict=True)
        if hasattr(text_out, 'pooler_output') and text_out.pooler_output is not None:
            tfeat = text_out.pooler_output
        else:
            tfeat = text_out.last_hidden_state[:, 0, :]
        t = self.text_proj(tfeat)
        t = self.act(t)

        # image features
        if hasattr(self.image_model, 'forward_features'):
            img_feat = self.image_model.forward_features(img)
            if img_feat.dim() > 2:
                img_feat = torch.flatten(img_feat, 1)
        else:
            img_feat = self.image_model(img)

        i = self.image_proj(img_feat)
        i = self.act(i)

        fused = torch.cat([t, i], dim=1)
        out = self.classifier(self.dropout(fused))
        return out

model = MultimodalClassifier(text_model, image_model, text_hidden_size, img_feat_dim, hidden_dim=512, num_classes=num_classes)
model = model.to(DEVICE)
print('Model created. Total params (approx):', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Loss & optimizer - with class weights for imbalance
labels = df['label'].values
class_counts = np.bincount(labels)
class_weights = torch.tensor(1.0 / (class_counts + 1e-8), dtype=torch.float).to(DEVICE)
print('Class counts:', class_counts, 'weights:', class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
from torch.cuda.amp import GradScaler, autocast
scaler = GradScaler()  # for mixed precision
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.7)

In [ ]:
# Training and evaluation functions
def train_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    losses = []
    all_preds, all_labels = [], []
    for enc, imgs, labels in tqdm(loader):
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                logits = model(enc, imgs)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(enc, imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.detach().cpu().numpy())
        losses.append(loss.item())
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1

def eval_epoch(model, loader, criterion):
    model.eval()
    losses = []
    all_preds, all_labels = [], []
    with torch.no_grad():
        for enc, imgs, labels in tqdm(loader):
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            with autocast():
                logits = model(enc, imgs)
                loss = criterion(logits, labels)
            preds = logits.argmax(dim=1).detach().cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.detach().cpu().numpy())
            losses.append(loss.item())
    avg_loss = np.mean(losses)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1, all_labels, all_preds

# Training loop with live plot data collection
history = {'train_loss':[], 'train_acc':[], 'train_f1':[], 'val_loss':[], 'val_acc':[], 'val_f1':[]}
best_val_f1 = 0.0
save_path = 'multimodal_emotion_best.pth'

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, criterion, scaler=scaler)
    val_loss, val_acc, val_f1, val_labels, val_preds = eval_epoch(model, val_loader, criterion)

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    print(f"Train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f}")
    print(f"Val   loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'model_state': model.state_dict(),
            'label_classes': le.classes_.tolist(),
            'tokenizer': PRETRAINED_TEXT,
            'config': {'IMG_SIZE': IMG_SIZE, 'MAX_TEXT_LEN': MAX_TEXT_LEN, 'IMAGE_BACKBONE': IMAGE_BACKBONE}
        }, save_path)
        print('Saved best model (f1 improved).')

# Plot training curves
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.plot(history['train_loss'], label='train_loss')
plt.plot(history['val_loss'], label='val_loss')
plt.legend(); plt.title('Loss')

plt.subplot(1,3,2)
plt.plot(history['train_acc'], label='train_acc')
plt.plot(history['val_acc'], label='val_acc')
plt.legend(); plt.title('Accuracy')

plt.subplot(1,3,3)
plt.plot(history['train_f1'], label='train_f1')
plt.plot(history['val_f1'], label='val_f1')
plt.legend(); plt.title('Macro F1')

plt.tight_layout()
plt.show()


In [ ]:
# Load best model and report
ckpt = torch.load(save_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
val_loss, val_acc, val_f1, val_labels, val_preds = eval_epoch(model, val_loader, criterion)
print('Final Val acc:', val_acc, 'f1:', val_f1)
print(classification_report(val_labels, val_preds, target_names=le.classes_))


In [ ]:
# Inference helper - pass a list of texts and corresponding PIL images
def predict(texts, pil_images):
    model.eval()
    enc = tokenizer(texts, padding='max_length', truncation=True, max_length=MAX_TEXT_LEN, return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    imgs = []
    for im in pil_images:
        i = image_transform(im).unsqueeze(0)
        imgs.append(i)
    imgs = torch.cat(imgs, dim=0).to(DEVICE)
    with torch.no_grad():
        logits = model(enc, imgs)
        preds = logits.argmax(dim=1).cpu().numpy()
    return le.inverse_transform(preds)

# Example usage:
# from PIL import Image
# txts = ['we live in a society...']
# imgs = [Image.open('img/23056.png').convert('RGB')]
# print(predict(txts, imgs))


In [ ]:
# Save label classes and quick metadata
import json
with open('label_classes.json', 'w') as f:
    json.dump({'classes': le.classes_.tolist()}, f)
print('Saved label_classes.json and model checkpoint:', save_path)
